# UI 및 컨트롤러 구현
- 학습이 완료된 모델을 import
- import한 모델을 바탕으로 사용자의 사진을 받아 결과를 리턴하는 3가지 서비스 메소드(전처리, Random Forest, K-Means) 구현
- 3개의 메소드를 잇는 UI 화면 및 컨트롤러 코드를 구현

## UI 및 컨트롤러 코드

In [ ]:
# 1. 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

# 2. 필수 라이브러리 설치
!pip install streamlit
!pip install streamlit ultralytics
!pip install streamlit ultralytics mediapipe
!npm install -g localtunnel
!pip install scikit-learn
!pip install ultralytics
!pip install mediapipe

%cd /content/drive/MyDrive/CV팀_PosePick

In [ ]:
%%writefile app.py

import streamlit as st
import numpy as np
from PIL import Image, ImageOps
import joblib
import time
from ultralytics import YOLO
import mediapipe as mp
import os
import cv2
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from sklearn.preprocessing import normalize  # ★ 코사인 우회를 위한 L2 정규화 라이브러리

# app.py 파일이 위치한 현재 폴더의 절대 경로를 자동으로 가져옴
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# --- [서버 시작 시 1회 로딩 및 대표 이미지 캐싱] ---

def load_kmeans_model(path):
    data = joblib.load(path) # pickle 대신 joblib 사용
    return data["scaler"], data["kmeans"]

def load_random_forest_model(path):
    data = joblib.load(path) # pickle 대신 joblib 사용
    return data["model"], data

@st.cache_resource
def load_models():
    # 1. 모델 경로 설정 및 로드 (요청하신 구버전 파일명으로 유지)
    kmeans_path = os.path.join(BASE_DIR, 'models', 'pose_kmeans_model.pkl')
    scaler, kmeans = load_kmeans_model(kmeans_path)

    rf_path = os.path.join(BASE_DIR, 'models', 'pose_random_forest_model_final.pkl')
    rf_model, rf_meta = load_random_forest_model(rf_path)

    # 2. 데이터셋 로드
    dataset_path = os.path.join(BASE_DIR, 'features', 'pose_final_dataset.npz')
    image_ids = np.load(dataset_path)['img_ids']

    scaled_path = os.path.join(BASE_DIR, 'features', 'good_train_scaled.npy')
    X_train_scaled = np.load(scaled_path)

    # 3. [최적화] 각 클러스터의 대표 이미지 인덱스(Centroid와 코사인 거리가 가장 가까운 학습 데이터)를 미리 계산
    X_train_normalized = normalize(X_train_scaled, norm='l2', axis=1)
    centroids = kmeans.cluster_centers_
    centroids_norm = normalize(centroids, norm='l2', axis=1)

    # 만약 피클 파일에 labels가 유실되었을 경우를 대비해 직접 predict 수행
    train_labels = kmeans.labels_ if hasattr(kmeans, 'labels_') else kmeans.predict(X_train_normalized)

    representatives = {}
    for k in range(kmeans.n_clusters):
        in_cluster_indices = np.where(train_labels == k)[0]
        if len(in_cluster_indices) == 0:
            continue

        cluster_feats = X_train_normalized[in_cluster_indices]
        centroid_k = centroids_norm[k]

        # 두 벡터가 정규화 상태이므로 내적값이 코사인 유사도
        cos_sims = np.dot(cluster_feats, centroid_k)

        best_local_idx = np.argmax(cos_sims)
        representatives[k] = in_cluster_indices[best_local_idx]

    # 4. YOLO & MediaPipe detector 초기화
    YOLO_PATH = os.path.join(BASE_DIR, 'models', 'yolov8m.pt')
    TASKS_PATH = os.path.join(BASE_DIR, 'models', 'pose_landmarker_heavy.task')

    yolo_model = YOLO(YOLO_PATH)
    base_options = python.BaseOptions(model_asset_path=TASKS_PATH)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False
    )
    detector = vision.PoseLandmarker.create_from_options(options)

    # (불필요한 diffs 중첩 루프 계산 제거로 서버 기동 속도를 대폭 개선)
    return scaler, representatives, kmeans, rf_model, rf_meta, image_ids, yolo_model, detector

# 모델 로드 (리턴값 개수 조정)
scaler, representatives, kmeans, rf_model, rf_meta, image_ids, yolo_model, detector = load_models()

# --- [firebase 인증] ---
import firebase_admin
from firebase_admin import credentials, firestore, storage

if not firebase_admin._apps:
    cred = os.path.join(BASE_DIR, 'security', 'firebase_key.json')
    cred = credentials.Certificate(cred)
    firebase_admin.initialize_app(cred, {
        'storageBucket': '<YOUR_FIREBASE_BUCKET>'
    })

db = firestore.client()
bucket = storage.bucket()

def get_image_url(image_id) -> str:
    """
    Firebase Storage가 발급한 다운로드 URL을 반환합니다.
    """
    try:
        blob = bucket.blob(f"selected/{image_id}.jpg")
        blob.make_public()
        return blob.public_url
    except Exception as e:
        bucket_name = "<YOUR_FIREBASE_BUCKET>"
        return f"https://firebasestorage.googleapis.com/v0/b/{bucket_name}/o/selected%2F{image_id}.jpg?alt=media"

# --- [동작 함수] ---

def infer_kmeans(X_input_scaled, kmeans, representatives, image_ids):
    """
    정규화 및 추론이 최적화된 KMeans 서비스 함수
    """
    # 1. 쿼리 피처 L2 정규화 (절대 생략 불가능!)
    X_input_normalized = normalize(X_input_scaled, norm='l2', axis=1)

    centroids = kmeans.cluster_centers_  # Shape: (20, 151)
    centroids_norm = normalize(centroids, norm='l2', axis=1)

    results = []
    for i in range(len(X_input_normalized)):
        query_normalized = X_input_normalized[i]

        # 2. 쿼리와 센트로이드들의 코사인 유사도 계산
        cos_sims = np.dot(centroids_norm, query_normalized)
        dists_cosine = 1.0 - cos_sims

        # 3. 소속 클러스터 및 거리 획득
        best_cluster_idx = np.argmin(dists_cosine)
        best_dist = float(dists_cosine[best_cluster_idx])

        # 4. 코사인 유사도 기반 점수 환산
        score = round(max(0.0, (1.0 - best_dist)) * 100.0, 2)

        # 5. 캐싱된 대표 이미지 인덱스 매칭
        best_idx = representatives.get(best_cluster_idx, 0)

        results.append({
            "image_id": image_ids[best_idx],
            "score":    score,
        })

    return results

# 점수 가공 (1회만 남겨둠)
def recalculate_score(results):
    if not results:
        return ()

    new_results = []
    for item in results:
        img_path = item[0]
        score = item[1]
        norm_score = round(float(score), 2) if score is not None else 0.0
        new_results.append((img_path, norm_score))

    return tuple(new_results)

# --- [유틸 함수] ---

def normalize_by_pelvis(X_features):
    X_norm = X_features.copy()
    coords_3d = X_norm[:, :88]
    N = coords_3d.shape[0]
    joints = coords_3d.reshape(N, 22, 4)
    left_hip  = joints[:, 12, :3]
    right_hip = joints[:, 13, :3]
    mid_hip   = (left_hip + right_hip) / 2.0
    joints[:, :, :3] -= mid_hip[:, np.newaxis, :]
    X_norm[:, :88] = joints.reshape(N, 88)
    return X_norm

def normalize_by_scale(X_features):
    X_norm = X_features.copy()
    coords_3d = X_norm[:, :88]
    N = coords_3d.shape[0]
    joints = coords_3d.reshape(N, 22, 4)

    left_shoulder  = joints[:, 0, :3]
    right_shoulder = joints[:, 1, :3]
    shoulder_width = np.linalg.norm(left_shoulder - right_shoulder, axis=1, keepdims=True)
    shoulder_width = np.maximum(shoulder_width, 1e-6)

    # 1. 3D 관절 좌표 정규화
    joints[:, :, :3] /= shoulder_width[:, np.newaxis, :]
    X_norm[:, :88] = joints.reshape(N, 88)

    # 2. [오류 수정] 3D 뼈대 거리 및 대칭성 피처 스케일 정규화
    shoulder_width_2d = shoulder_width.reshape(N, 1)
    X_norm[:, 88:114] /= shoulder_width_2d
    X_norm[:, 148:151] /= shoulder_width_2d

    return X_norm

def scale_input(filtered_feature_list):
    X_input = np.array([
        np.concatenate([
            d['pose_coords_3d_vis'],
            d['pose_dist_3d'],
            d['pose_ratios'],
            d['pose_angles'],
            d['pose_symmetry']
        ]) for d in filtered_feature_list
    ])
    X_input = normalize_by_pelvis(X_input)
    X_input = normalize_by_scale(X_input)
    return scaler.transform(X_input)

# 25개 뼈대 연결 구성
POSE_EDGES_CONFIG = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16), (15, 17), (15, 19), (15, 21),
    (11, 23), (12, 24), (23, 24), (23, 25), (25, 27), (24, 26), (26, 28), (27, 29),
    (28, 30), (29, 31), (30, 32), (27, 31), (28, 32), (16, 18), (16, 20), (16, 22),
    (17, 19)
]

def enhance_image_for_mediapipe(img_np):
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    enhanced = cv2.merge((cl, a, b))
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)

    gamma = 1.5
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(enhanced, table)

def yolo_crop_and_resize(image_np, target_size=(1024, 1024)):
    h, w, _ = image_np.shape
    results = yolo_model.predict(image_np, classes=[0], verbose=False)

    if len(results[0].boxes) == 0:
        side = min(w, h)
        xmin, ymin = (w - side) // 2, (h - side) // 2
        return cv2.resize(image_np[ymin:ymin+side, xmin:xmin+side], target_size), {"xmin": xmin, "ymin": ymin, "cw": side, "ch": side}

    box = results[0].boxes[0].xyxy[0].cpu().numpy()
    x1, y1, x2, y2 = map(int, box)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * 1.5

    xmin, ymin = int(cx - side / 2), int(cy - side / 2)
    crop_info = {"xmin": xmin, "ymin": ymin, "cw": int(side), "ch": int(side)}

    pad_xmin, pad_ymin = max(0, -xmin), max(0, -ymin)
    pad_xmax, pad_ymax = max(0, int(cx + side / 2) - w), max(0, int(cy + side / 2) - h)
    img_padded = cv2.copyMakeBorder(image_np, pad_ymin, pad_ymax, pad_xmin, pad_xmax, cv2.BORDER_CONSTANT, value=[0, 0, 0])

    c_xmin, c_ymin = xmin + pad_xmin, ymin + pad_ymin
    image_cropped = img_padded[c_ymin:c_ymin+int(side), c_xmin:c_xmin+int(side)]
    return cv2.resize(image_cropped, target_size, interpolation=cv2.INTER_AREA), crop_info

def extract_features_from_task(detection_result, img_id):
    if not detection_result or not detection_result.pose_landmarks:
        return {"img_id": str(img_id), "features": {"coords_3d_vis": [0.0]*88, "dist_3d_vis": [0.0]*25}}

    landmarks = detection_result.pose_landmarks[0]
    coords_88, pts, vis = [], {}, {}

    for i in range(11, 33):
        lm = landmarks[i]
        v = getattr(lm, 'visibility', 0.0)
        coords_88.extend([lm.x, lm.y, lm.z, v])
        pts[i] = np.array([lm.x, lm.y, lm.z])
        vis[i] = v

    dist_3d_vis = []
    for p1, p2 in POSE_EDGES_CONFIG:
        dist = float(np.linalg.norm(pts[p1] - pts[p2]))
        dist_3d_vis.append(dist)

    return {"img_id": str(img_id), "features": {"coords_3d_vis": coords_88, "dist_3d_vis": dist_3d_vis}}

def calculate_angle_rad(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return float(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

def process_pose_features(json_data_list):
    processed_list = []

    for item in json_data_list:
        img_id = item['img_id']
        coords_all = item['features']['coords_3d_vis']
        dist_vis_all = item['features']['dist_3d_vis']

        pts_3d = {}
        pts_2d = {}
        for i in range(11, 33):
            idx = (i - 11) * 4
            x, y, z = coords_all[idx], coords_all[idx+1], coords_all[idx+2]
            pts_3d[i] = np.array([x, y, z])
            pts_2d[i] = np.array([x, y])

        dist_3d = [dist_vis_all[i] for i in range(25)]

        ratios = []
        for i, (p1, p2) in enumerate(POSE_EDGES_CONFIG):
            d2d = np.linalg.norm(pts_2d[p1] - pts_2d[p2])
            d3d = dist_3d[i] + 1e-6
            ratios.append(float(d2d / d3d))

        angles_rad = [
            calculate_angle_rad(pts_3d[11], pts_3d[13], pts_3d[15]),
            calculate_angle_rad(pts_3d[12], pts_3d[14], pts_3d[16]),
            calculate_angle_rad(pts_3d[23], pts_3d[25], pts_3d[27]),
            calculate_angle_rad(pts_3d[24], pts_3d[26], pts_3d[28]),
            calculate_angle_rad(pts_3d[13], pts_3d[11], pts_3d[23]),
            calculate_angle_rad(pts_3d[14], pts_3d[12], pts_3d[24]),
            calculate_angle_rad(pts_3d[11], pts_3d[23], pts_3d[25]),
            calculate_angle_rad(pts_3d[12], pts_3d[24], pts_3d[26])
        ]

        symmetry_diff = [
            abs(pts_3d[11][1] - pts_3d[12][1]),
            abs(pts_3d[13][1] - pts_3d[14][1]),
            abs(pts_3d[15][1] - pts_3d[16][1]),
            abs(pts_3d[23][1] - pts_3d[24][1]),
            abs(pts_3d[25][1] - pts_3d[26][1])
        ]

        processed_item = {
            "img_id": img_id,
            "pose_coords_3d_vis": coords_all,
            "pose_dist_3d": dist_3d,
            "pose_ratios": ratios,
            "pose_angles": angles_rad,
            "pose_symmetry": symmetry_diff
        }
        processed_list.append(processed_item)

    return processed_list

def draw_skeleton_on_np(image_np, landmarks_88, crop_info):
    canvas = image_np.copy()
    pts = {}
    for i in range(11, 33):
        idx = (i - 11) * 4
        if landmarks_88[idx + 3] > 0.3:
            rx = int(landmarks_88[idx] * crop_info['cw'] + crop_info['xmin'])
            ry = int(landmarks_88[idx+1] * crop_info['ch'] + crop_info['ymin'])
            pts[i] = (rx, ry)
            cv2.circle(canvas, (rx, ry), 10, (255, 0, 0), -1)

    for p1, p2 in POSE_EDGES_CONFIG:
        if p1 in pts and p2 in pts:
            cv2.line(canvas, pts[p1], pts[p2], (0, 255, 0), 5)
    return canvas

def run_posepick_service_final(raw_images_list):
    final_json = []
    skeleton_results = []

    for i, raw_np in enumerate(raw_images_list):
        current_id = str(i)

        img_temp = np.array(ImageOps.exif_transpose(Image.fromarray(raw_np)))

        img_input, c_info = yolo_crop_and_resize(img_temp)
        img_enhanced = enhance_image_for_mediapipe(img_input)

        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_enhanced)
        pose_results = detector.detect(mp_image)

        data = extract_features_from_task(pose_results, current_id)
        final_json.append(data)

        skel_img = draw_skeleton_on_np(img_temp, data['features']['coords_3d_vis'], c_info)
        skeleton_results.append(skel_img)

    return final_json, skeleton_results

# --- [팀원들이 작성할 AI 모듈 위치] ---

def preprocess(image_array_list):
    json_data_list, skeletons = run_posepick_service_final(image_array_list)
    final_processed_data = process_pose_features(json_data_list)
    time.sleep(2)
    return final_processed_data

def build_rf_input(feature_list):
    X_input = np.array([
        np.concatenate([
            d['pose_coords_3d_vis'],
            d['pose_dist_3d'],
            d['pose_ratios'],
            d['pose_angles'],
            d['pose_symmetry']
        ]) for d in feature_list
    ], dtype=np.float32)

    X_input = np.nan_to_num(X_input, nan=0.0, posinf=0.0, neginf=0.0)

    if X_input.shape[1] != 151:
        raise ValueError(f"RF 입력 feature 차원이 다릅니다: {X_input.shape[1]}")

    return X_input

RF_GOOD_THRESHOLD = 0.5

def predict_good_pose(feature_list):
    if not feature_list:
        return []

    X_input = build_rf_input(feature_list)

    expected_dim = rf_meta.get("n_features")
    if expected_dim is not None and X_input.shape[1] != expected_dim:
        raise ValueError(
            f"RF 입력 feature 차원이 다릅니다: "
            f"input={X_input.shape[1]}, expected={expected_dim}"
        )

    proba = rf_model.predict_proba(X_input)

    return [
        1 if float(prob[1]) >= RF_GOOD_THRESHOLD else 0
        for prob in proba
    ]

def analyze_similarity(filtered_feature_list):
    X_input_scaled = scale_input(filtered_feature_list)
    # [최적화]Representatives 캐싱 딕셔너리를 활용하여 초고속 매칭 수행
    results = infer_kmeans(X_input_scaled, kmeans, representatives, image_ids)
    return [(get_image_url(r['image_id']), r['score']) for r in results]

# --- [UI 세션 관리] ---
if 'page' not in st.session_state:
    st.session_state.page = 'home'
if 'results' not in st.session_state:
    st.session_state.results = None
if 'selected_idx' not in st.session_state:
    st.session_state.selected_idx = None

# --- [공통 상단 UI] ---
st.set_page_config(page_title="PosePick", layout="wide")
st.title("📸 PosePick")
st.markdown("---")

# --- [화면 1 & 2 & 3: 홈 및 서비스 실행] ---
if st.session_state.page == 'home':
    st.subheader("당신의 사진 중에서 가장 비율이 잘 나온 베스트 컷을 찾아드려요.")
    st.markdown("최대 50장의 전신 연사 사진을 업로드해 보세요. AI가 뼈대 좌표와 포즈를 분석합니다.")

    uploaded_files = st.file_uploader("사진을 추가해 주세요 (최대 50장)", accept_multiple_files=True, type=['jpg', 'jpeg', 'png'])

    if uploaded_files:
        if st.button("서비스 시작", use_container_width=True):
            with st.spinner("AI가 뼈대를 분석 중입니다..."):
                progress_bar = st.progress(0)
                img_arrays = [np.array(ImageOps.exif_transpose(Image.open(f)).convert("RGB")) for f in uploaded_files]
                progress_bar.progress(20)

                all_features = preprocess(img_arrays)
                progress_bar.progress(50)

                rf_results = predict_good_pose(all_features)
                progress_bar.progress(70)

                passed_indices = [i for i, res in enumerate(rf_results) if res == 1]

                if not passed_indices:
                    st.session_state.page = 'no_result'
                    st.rerun()

                passed_features = [all_features[i] for i in passed_indices]
                analysis_output = analyze_similarity(passed_features)
                analysis_output = recalculate_score(analysis_output)

                final_results = []
                for i, (m_path, score) in enumerate(analysis_output):
                    final_results.append({
                        'user_img':        uploaded_files[passed_indices[i]],
                        'score':           score,
                        'model_img_path':  m_path
                    })

                st.session_state.results = sorted(final_results, key=lambda x: x['score'], reverse=True)
                st.session_state.page = 'result'
                st.rerun()

# --- [화면 3: 통과된 사진 없음] ---
elif st.session_state.page == 'no_result':
    st.warning("⚠️ 분석 결과, 추천할 만한 베스트 컷이 없습니다.")
    if st.button("🏠 홈으로 돌아가기"):
        st.session_state.page = 'home'
        st.rerun()

# --- [화면 4: 분석 완료 화면] ---
elif st.session_state.page == 'result':
    st.subheader("🏆 분석 완료! 당신의 베스트 컷입니다.")

    cols = st.columns(3)
    for i in range(min(3, len(st.session_state.results))):
        with cols[i]:
            st.success(f"🥇 TOP {i+1} (점수: {st.session_state.results[i]['score']}점)")
            st.image(st.session_state.results[i]['user_img'], use_container_width=True)
            if st.button(f"🔍 TOP {i+1} 자세히 보기", key=f"btn_top_{i}", use_container_width=True):
                st.session_state.selected_idx = i
                st.session_state.page = 'detail'
                st.rerun()

    if len(st.session_state.results) > 3:
        st.markdown("---")
        st.subheader("📍 그 외 통과된 사진들")
        rest_results = st.session_state.results[3:]
        cols_rest = st.columns(4)
        for idx, res in enumerate(rest_results):
            actual_idx = idx + 3
            with cols_rest[idx % 4]:
                st.image(res['user_img'], use_container_width=True)
                if st.button(f"🔍 자세히 보기", key=f"btn_rest_{actual_idx}", use_container_width=True):
                    st.session_state.selected_idx = actual_idx
                    st.session_state.page = 'detail'
                    st.rerun()

    st.markdown("---")
    if st.button("🏠 처음부터 다시하기 (홈으로)", use_container_width=True):
        st.session_state.page = 'home'
        st.rerun()

# --- [화면 5: 상세 분석 화면] ---
elif st.session_state.page == 'detail':
    res = st.session_state.results[st.session_state.selected_idx]
    rank = st.session_state.selected_idx + 1
    st.subheader(f"🔍 {rank}위 포즈 상세 비교 (점수: {res['score']}점)")

    col1, col2 = st.columns(2)
    with col1:
        st.write("🚩 내 포즈")
        st.image(res['user_img'], use_container_width=True)
    with col2:
        st.write("⭐ 이상적인 모델 포즈")
        st.image(res['model_img_path'], use_container_width=True)

    st.markdown("---")
    col_nav1, col_nav2 = st.columns(2)
    with col_nav1:
        if st.button("⬅️ 결과 목록으로 돌아가기", use_container_width=True):
            st.session_state.page = 'result'
            st.rerun()
    with col_nav2:
        if st.button("🏠 홈으로 돌아가기", use_container_width=True):
            st.session_state.page = 'home'
            st.rerun()

In [ ]:
import os
import time

# 1. 기존에 켜져있는 스트림릿 종료 (충돌 방지)
os.system("fuser -k 8501/tcp")

# 2. 스트림릿 백그라운드 실행
os.system("nohup streamlit run app.py > logs.txt 2>&1 &")
time.sleep(3)

# 3. 클라우드플레어 터널 다운로드 및 실행
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("\n=========================================================")
print("⭐ 아래 출력되는 로그 중에서 'https://... .trycloudflare.com' 링크를 클릭해주세요! ⭐")
print("=========================================================\n")

# 터널 열기
!./cloudflared tunnel --url http://localhost:8501